# Detección con YOLOv8 — Gestos de mano y Casco de seguridad

En este taller entrenamos dos detectores usando **YOLOv8** (You Only Look Once), el modelo de detección de objetos en tiempo real más popular actualmente.

**Casos de uso:**
1. **Gestos de mano** — pulgar arriba 👍, pulgar abajo 👎, más o menos 👋
2. **Casco de seguridad** — persona con casco / persona sin casco

> **Google Colab:** Activa la GPU antes de empezar: *Entorno de ejecución → Cambiar tipo → T4 GPU*

## ⚙️ Instalación y configuración

In [ ]:
!pip install -q ultralytics kagglehub

import sys, os, glob, random, shutil, yaml
import xml.etree.ElementTree as ET
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
from ultralytics import YOLO
import kagglehub

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR  = '/content' if IN_COLAB else '.'

print(f"Entorno : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

## 🛠️ Utilidades de conversión de datasets

Los datasets de Kaggle pueden venir en distintos formatos. Estas funciones los convierten al formato que necesita YOLO:
- **Pascal VOC (XML)** → YOLO txt
- **Carpetas por clase** (clasificación) → YOLO txt con bounding box = imagen completa

In [ ]:
def voc_xml_to_yolo(xml_path, class_names):
    """Convierte anotación Pascal VOC (XML) a líneas en formato YOLO."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size  = root.find('size')
    img_w = float(size.find('width').text)
    img_h = float(size.find('height').text)

    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in class_names:
            continue
        cid  = class_names.index(name)
        bbox = obj.find('bndbox')
        xmin = float(bbox.find('xmin').text)
        ymin = float(bbox.find('ymin').text)
        xmax = float(bbox.find('xmax').text)
        ymax = float(bbox.find('ymax').text)
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w  = (xmax - xmin) / img_w
        h  = (ymax - ymin) / img_h
        lines.append(f"{cid} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines


def clasificacion_a_yolo(src_dir, dst_dir, class_names, splits=(0.8, 0.1, 0.1)):
    """
    Convierte un dataset de clasificación (carpetas por clase) a formato YOLO.
    El bounding box es la imagen completa (útil cuando la mano llena el frame).
    """
    for split in ['train', 'valid', 'test']:
        os.makedirs(os.path.join(dst_dir, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(dst_dir, split, 'labels'), exist_ok=True)

    all_items = []
    for cid, cls in enumerate(class_names):
        folder = os.path.join(src_dir, cls)
        if not os.path.isdir(folder):
            continue
        imgs = glob.glob(os.path.join(folder, '*.jpg')) + \
               glob.glob(os.path.join(folder, '*.png'))
        for img in imgs:
            all_items.append((img, cid))

    random.shuffle(all_items)
    n = len(all_items)
    n_train = int(n * splits[0])
    n_valid = int(n * splits[1])
    split_data = {
        'train': all_items[:n_train],
        'valid': all_items[n_train:n_train + n_valid],
        'test' : all_items[n_train + n_valid:],
    }

    for split, items in split_data.items():
        for img_path, cid in items:
            fname = os.path.basename(img_path)
            stem  = os.path.splitext(fname)[0]
            shutil.copy(img_path, os.path.join(dst_dir, split, 'images', fname))
            label = f"{cid} 0.500000 0.500000 1.000000 1.000000"
            with open(os.path.join(dst_dir, split, 'labels', stem + '.txt'), 'w') as f:
                f.write(label + '\n')

    print(f"  train : {len(split_data['train'])} imágenes")
    print(f"  valid : {len(split_data['valid'])} imágenes")
    print(f"  test  : {len(split_data['test'])} imágenes")


def crear_data_yaml(dst_dir, class_names):
    cfg = {
        'path'  : dst_dir,
        'train' : 'train/images',
        'val'   : 'valid/images',
        'test'  : 'test/images',
        'nc'    : len(class_names),
        'names' : class_names,
    }
    out = os.path.join(dst_dir, 'data.yaml')
    with open(out, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return out


print("Utilidades de conversión listas.")

---
# Parte 1 — Detección de gestos de mano

Entrenamos YOLOv8 para reconocer tres gestos:
- 👍 `thumbs_up` — pulgar arriba
- 👎 `thumbs_down` — pulgar abajo
- ✋ `thumbs_sideways` — más o menos

Dataset: **Sign Language for Numbers** (`muhammadkhalid/sign-language-for-numbers`) — Kaggle

In [ ]:
DST_GESTOS = os.path.join(BASE_DIR, 'datasets', 'gestos')

if os.path.exists(os.path.join(DST_GESTOS, 'data.yaml')):
    print("Dataset de gestos ya preparado.")
else:
    # Descargar desde Kaggle
    print("Descargando dataset de gestos desde Kaggle...")
    raw_path = kagglehub.dataset_download("muhammadkhalid/sign-language-for-numbers")
    print("Descargado en:", raw_path)

    # Mostrar estructura para identificar carpetas de clases
    print("\nEstructura del dataset:")
    for root, dirs, files in os.walk(raw_path):
        level = root.replace(raw_path, '').count(os.sep)
        if level < 3:
            print('  ' * level + os.path.basename(root) + '/')

    # Buscar la carpeta raíz con subcarpetas de clases
    src_dir = None
    for root, dirs, _ in os.walk(raw_path):
        if len(dirs) >= 3 and all(
            len(glob.glob(os.path.join(root, d, '*.jpg')) +
                glob.glob(os.path.join(root, d, '*.png'))) > 0
            for d in dirs[:3]
        ):
            src_dir = root
            break

    if src_dir is None:
        src_dir = raw_path

    clases_disponibles = sorted([
        d for d in os.listdir(src_dir)
        if os.path.isdir(os.path.join(src_dir, d))
    ])
    print(f"\nClases disponibles ({len(clases_disponibles)}):", clases_disponibles[:15])

    # Seleccionar las clases que corresponden a los tres gestos
    # Ajusta estos nombres según lo que muestre la celda anterior
    CLASES_GESTOS = ['thumbs_up', 'thumbs_down', 'thumbs_sideways']

    # Filtrar clases que existan en el dataset
    CLASES_GESTOS = [c for c in CLASES_GESTOS if c in clases_disponibles]
    if not CLASES_GESTOS:
        # Si no coinciden exactamente, usar las primeras 3 clases disponibles
        CLASES_GESTOS = clases_disponibles[:3]
        print(f"Clases exactas no encontradas. Usando: {CLASES_GESTOS}")

    print(f"\nConvirtiendo a formato YOLO con clases: {CLASES_GESTOS}")
    clasificacion_a_yolo(src_dir, DST_GESTOS, CLASES_GESTOS)
    crear_data_yaml(DST_GESTOS, CLASES_GESTOS)
    print("Listo.")

In [ ]:
with open(os.path.join(DST_GESTOS, 'data.yaml')) as f:
    data_cfg = yaml.safe_load(f)

print("Clases:", data_cfg['names'])
for split in ['train', 'valid', 'test']:
    path = os.path.join(DST_GESTOS, split, 'images')
    if os.path.exists(path):
        print(f"{split:6s}: {len(os.listdir(path))} imágenes")

# Referencia para el entrenamiento
dataset_gestos_yaml = os.path.join(DST_GESTOS, 'data.yaml')

### Entrenamiento — Gestos

Usamos **YOLOv8n** (nano), el modelo más pequeño y rápido. Es ideal para empezar y funciona bien con datasets pequeños.

| Modelo | Tamaño | Velocidad | Precisión |
|--------|--------|-----------|----------|
| YOLOv8n | ~6 MB | Muy rápido | Buena |
| YOLOv8s | ~22 MB | Rápido | Mejor |
| YOLOv8m | ~52 MB | Medio | Muy buena |

In [ ]:
model_gestos = YOLO('yolov8n.pt')

results_gestos = model_gestos.train(
    data=dataset_gestos_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    name='gestos_yolov8n',
    project=os.path.join(BASE_DIR, 'runs', 'gestos'),
    patience=10,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=False,
)

run_dir_gestos       = results_gestos.save_dir
best_weights_gestos  = os.path.join(run_dir_gestos, 'weights', 'best.pt')
print("Entrenamiento completado.")
print(f"mAP@50: {results_gestos.results_dict.get('metrics/mAP50(B)', 'N/A')}")

In [ ]:
# Visualizar curvas de entrenamiento
run_dir = results_gestos.save_dir
curve_path = os.path.join(run_dir, 'results.png')

if os.path.exists(curve_path):
    display(Image(filename=curve_path, width=900))

### Evaluación — Gestos

In [ ]:
model_gestos_best = YOLO(best_weights_gestos)
metrics = model_gestos_best.val(data=dataset_gestos_yaml, verbose=False)
d = metrics.results_dict
print(f"Precisión (P) : {d['metrics/precision(B)']:.4f}")
print(f"Recall    (R) : {d['metrics/recall(B)']:.4f}")
print(f"mAP@50        : {d['metrics/mAP50(B)']:.4f}")
print(f"mAP@50-95     : {d['metrics/mAP50-95(B)']:.4f}")

### Predicciones — Gestos

Corremos inferencia sobre las imágenes de prueba y visualizamos los resultados.

In [ ]:
import glob, random

test_images = glob.glob(os.path.join(DST_GESTOS, 'test', 'images', '*.jpg'))
if not test_images:
    test_images = glob.glob(os.path.join(DST_GESTOS, 'valid', 'images', '*.jpg'))

sample = random.sample(test_images, min(6, len(test_images)))
resultados = model_gestos_best.predict(source=sample, conf=0.4, save=True, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, r in zip(axes.flat, resultados):
    img_rgb = r.plot()[..., ::-1]
    labels  = [r.names[int(c)] for c in r.boxes.cls] if r.boxes else ['Sin detección']
    ax.imshow(img_rgb)
    ax.set_title(', '.join(labels), fontsize=9)
    ax.axis('off')

plt.suptitle('Predicciones — Gestos de mano', fontsize=13)
plt.tight_layout()
plt.show()

---
# Parte 2 — Detección de casco de seguridad

Entrenamos YOLOv8 para detectar:
- ✅ `helmet` — persona con casco
- ❌ `head` — persona sin casco

Dataset: **Hard Hat Detection** (`andrewmvd/hard-hat-detection`) — Kaggle  
Formato original: Pascal VOC (XML) → convertido automáticamente a YOLO.

In [ ]:
DST_CASCO   = os.path.join(BASE_DIR, 'datasets', 'casco')
CLASES_CASCO = ['helmet', 'head']   # head = sin casco

if os.path.exists(os.path.join(DST_CASCO, 'data.yaml')):
    print("Dataset de cascos ya preparado.")
else:
    print("Descargando dataset de cascos desde Kaggle...")
    raw_path = kagglehub.dataset_download("andrewmvd/hard-hat-detection")
    print("Descargado en:", raw_path)

    # Buscar imágenes y anotaciones XML
    imagenes   = glob.glob(os.path.join(raw_path, '**', '*.png'), recursive=True) + \
                 glob.glob(os.path.join(raw_path, '**', '*.jpg'), recursive=True)
    xml_files  = glob.glob(os.path.join(raw_path, '**', '*.xml'), recursive=True)
    print(f"Imágenes encontradas : {len(imagenes)}")
    print(f"Archivos XML         : {len(xml_files)}")

    # Crear estructura de carpetas
    for split in ['train', 'valid', 'test']:
        os.makedirs(os.path.join(DST_CASCO, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(DST_CASCO, split, 'labels'), exist_ok=True)

    # Dividir 80/10/10
    random.shuffle(imagenes)
    n       = len(imagenes)
    n_train = int(n * 0.8)
    n_valid = int(n * 0.1)
    splits  = {
        'train': imagenes[:n_train],
        'valid': imagenes[n_train:n_train + n_valid],
        'test' : imagenes[n_train + n_valid:],
    }

    # Mapa de XML por nombre de archivo
    xml_map = {os.path.splitext(os.path.basename(x))[0]: x for x in xml_files}

    for split, imgs in splits.items():
        for img_path in imgs:
            stem  = os.path.splitext(os.path.basename(img_path))[0]
            fname = os.path.basename(img_path)
            shutil.copy(img_path, os.path.join(DST_CASCO, split, 'images', fname))
            if stem in xml_map:
                lines = voc_xml_to_yolo(xml_map[stem], CLASES_CASCO)
                if lines:
                    with open(os.path.join(DST_CASCO, split, 'labels', stem + '.txt'), 'w') as f:
                        f.write('\n'.join(lines) + '\n')

    crear_data_yaml(DST_CASCO, CLASES_CASCO)
    for split, imgs in splits.items():
        print(f"  {split:6s}: {len(imgs)} imágenes")
    print("Listo.")

dataset_casco_yaml = os.path.join(DST_CASCO, 'data.yaml')

### Entrenamiento — Casco

In [ ]:
model_casco = YOLO('yolov8n.pt')

results_casco = model_casco.train(
    data=dataset_casco_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    name='casco_yolov8n',
    project=os.path.join(BASE_DIR, 'runs', 'casco'),
    patience=10,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=False,
)

run_dir_casco      = results_casco.save_dir
best_weights_casco = os.path.join(run_dir_casco, 'weights', 'best.pt')
print("Entrenamiento completado.")
print(f"mAP@50: {results_casco.results_dict.get('metrics/mAP50(B)', 'N/A')}")

In [ ]:
run_dir_casco = results_casco.save_dir
curve_path_casco = os.path.join(run_dir_casco, 'results.png')

if os.path.exists(curve_path_casco):
    display(Image(filename=curve_path_casco, width=900))

### Evaluación — Casco

In [ ]:
model_casco_best = YOLO(best_weights_casco)
metrics_casco = model_casco_best.val(data=dataset_casco_yaml, verbose=False)
d = metrics_casco.results_dict
print(f"Precisión (P) : {d['metrics/precision(B)']:.4f}")
print(f"Recall    (R) : {d['metrics/recall(B)']:.4f}")
print(f"mAP@50        : {d['metrics/mAP50(B)']:.4f}")
print(f"mAP@50-95     : {d['metrics/mAP50-95(B)']:.4f}")

### Predicciones — Casco

In [ ]:
def buscar_imagenes(directorio):
    """Busca imágenes jpg y png en un directorio."""
    imgs = glob.glob(os.path.join(directorio, '*.jpg')) + \
           glob.glob(os.path.join(directorio, '*.png'))
    return imgs

test_images_casco = buscar_imagenes(os.path.join(DST_CASCO, 'test', 'images'))
if not test_images_casco:
    test_images_casco = buscar_imagenes(os.path.join(DST_CASCO, 'valid', 'images'))

print(f"Imágenes encontradas: {len(test_images_casco)}")
if not test_images_casco:
    print("No se encontraron imágenes. Estructura actual:")
    for root, dirs, files in os.walk(DST_CASCO):
        level = root.replace(DST_CASCO, '').count(os.sep)
        if level < 3:
            print('  ' * level + os.path.basename(root) + f'/ ({len(files)} archivos)')
else:
    sample_casco     = random.sample(test_images_casco, min(6, len(test_images_casco)))
    resultados_casco = model_casco_best.predict(source=sample_casco, conf=0.4, verbose=False)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, r in zip(axes.flat, resultados_casco):
        img_rgb = r.plot()[..., ::-1]
        labels  = [r.names[int(c)] for c in r.boxes.cls] if r.boxes else ['Sin detección']
        tiene_casco = any('helmet' in l.lower() and 'no' not in l.lower() for l in labels)
        color = 'green' if tiene_casco else 'red'
        ax.imshow(img_rgb)
        ax.set_title(', '.join(labels) if labels else 'Sin detección', fontsize=9, color=color)
        ax.axis('off')

    plt.suptitle('Predicciones — Casco (verde=con casco, rojo=sin casco)', fontsize=12)
    plt.tight_layout()
    plt.show()

---
# Parte 3 — Demo combinado con webcam (solo Colab)

Capturamos una foto desde la webcam y corremos ambos modelos sobre ella.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2

# Paso 1: registrar la función JS
display(Javascript('''
    async function tomarFoto(quality) {
        const div = document.createElement('div');
        const btn = document.createElement('button');
        btn.textContent = '📷 Tomar foto';
        btn.style.fontSize = '18px';
        btn.style.padding = '10px';
        div.appendChild(btn);

        const video = document.createElement('video');
        video.style.display = 'block';
        video.style.marginTop = '10px';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => btn.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width  = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
    }
'''))

# Paso 2: llamar la función y obtener la imagen
data      = eval_js('tomarFoto(0.8)')
binary    = b64decode(data.split(',')[1])
img_array = np.frombuffer(binary, dtype=np.uint8)
frame     = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
foto_path = '/content/webcam_foto.jpg'
cv2.imwrite(foto_path, frame)
print("Foto capturada correctamente.")

# Paso 3: correr ambos modelos sobre la foto
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (model, titulo) in zip(axes, [
    (model_gestos_best, 'Gestos de mano'),
    (model_casco_best,  'Casco de seguridad')
]):
    res    = model.predict(source=foto_path, conf=0.35, verbose=False)[0]
    labels = [res.names[int(c)] for c in res.boxes.cls] if res.boxes else ['Sin detección']
    ax.imshow(res.plot()[..., ::-1])
    ax.set_title(f"{titulo}\n{', '.join(labels)}", fontsize=11)
    ax.axis('off')

plt.suptitle('Demo webcam', fontsize=13)
plt.tight_layout()
plt.show()

## 💾 Guardar los modelos entrenados

In [ ]:
import shutil

save_dir = os.path.join(BASE_DIR, 'modelos_entrenados')
os.makedirs(save_dir, exist_ok=True)

shutil.copy(best_weights_gestos, os.path.join(save_dir, 'gestos_best.pt'))
shutil.copy(best_weights_casco,  os.path.join(save_dir, 'casco_best.pt'))
print(f"Modelos guardados en: {save_dir}")

# Descargar a tu computadora si estás en Colab
if IN_COLAB:
    from google.colab import files
    files.download(os.path.join(save_dir, 'gestos_best.pt'))
    files.download(os.path.join(save_dir, 'casco_best.pt'))

## Conclusión

En este taller entrenamos dos detectores de objetos con **YOLOv8**:

| Modelo | Clases | Técnica |
|--------|--------|---------|
| Gestos de mano | thumbs_up, thumbs_down, más o menos | Transfer learning desde COCO |
| Casco de seguridad | helmet, no_helmet | Transfer learning desde COCO |

**Conceptos clave aprendidos:**
- **Transfer learning con YOLO**: partimos de `yolov8n.pt` (preentrenado en COCO con 80 clases) y lo adaptamos a nuestras clases específicas.
- **mAP (mean Average Precision)**: métrica estándar para evaluar detectores de objetos.
- **Confianza (conf)**: umbral para filtrar detecciones — valores más altos = menos falsos positivos pero más falsos negativos.
- **Early stopping**: detiene el entrenamiento automáticamente si el modelo deja de mejorar.

**Próximos pasos posibles:**
- Usar `yolov8s.pt` o `yolov8m.pt` para mayor precisión
- Aumentar las épocas de entrenamiento
- Agregar más clases de gestos
- Implementar detección en video en tiempo real